# S6E10 | Auxiliary-Task Features | LB 0.96129

> **Part 4 of a series.** [Part 1: Route-ID FE + OG-Model Stack (LB 0.96108)](https://www.kaggle.com/code/goodpjw2008/s6e10-route-id-fe-og-model-stack-lb-0-96108) found the two feature ideas. [Part 2: Two-View GBDT + RealMLP Stack (LB 0.96125)](https://www.kaggle.com/code/goodpjw2008/s6e10-two-view-gbdt-realmlp-stack-lb-0-96125) added a second feature view and a RealMLP schedule. [Part 3: CatBoost CTR + GLM-Margin XGB (LB 0.96127)](https://www.kaggle.com/code/goodpjw2008/s6e10-catboost-ctr-glm-margin-xgb-lb-0-96127) added two tree variants and measured why the score had stopped moving. This part adds one new kind of feature: **predict every column from the other columns, and give those predictions to the label model.**

**Public LB 0.96129 · nested 5-fold CV 0.96167** — 29 models on one shared fold split, combined by logistic regression on logits.

**The whole notebook in one picture.** The numbers are from our local run, and each panel names the section that derives them. The drawing code is in the hidden cell.

In [ ]:
# The whole notebook in one picture.  Every number is from our local run and is derived in
# the section named in each panel; nothing here is computed by this cell.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

INK, MUTED, RULE, CARD = "#1f2937", "#6b7280", "#d1d5db", "#f3f4f6"
SEEN, REAL, NEW = "#f59e0b", "#2563eb", "#059669"  # one model in CV / the stack or the leaderboard / new here
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "text.color": INK,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED})
fig = plt.figure(figsize=(12, 11.5), dpi=110, facecolor="white")
fig.text(0.04, 0.972, "Part 4 in one picture", fontsize=23, fontweight="bold", va="center")
fig.text(0.04, 0.944, "One new idea: predict every column from the other 20, and give the predictions "
         "to the model that predicts the label.", fontsize=12.5, color=MUTED, va="center")

# ---------------------------------------------------------------- the idea
ax = fig.add_axes([0.04, 0.80, 0.93, 0.12]); ax.set_xlim(0, 100); ax.set_ylim(0, 10); ax.axis("off")
boxes = [(0.0, 15.0, "ALL ROWS", "1M train + test rows\nno label used", CARD, INK),
         (19.0, 20.0, "16 AUXILIARY TASKS", "hide one column, predict it\nfrom the other 20", "#d1fae5", INK),
         (43.0, 21.0, "NEW FEATURES", "the expected value\nof each rating", "#d1fae5", INK),
         (68.0, 17.0, "29 MODELS", "26 from Part 3\n+ 3 trees that use them", CARD, INK),
         (89.0, 11.0, "RESULT", "CV 0.96167\nLB 0.96129", "#dbeafe", INK)]
for x, w, head, body, col, tc in boxes:
    ax.add_patch(FancyBboxPatch((x, 0.6), w, 8.8, boxstyle="round,pad=0,rounding_size=0.9", fc=col, ec="none"))
    ax.text(x + w / 2, 7.6, head, ha="center", va="center", fontsize=10.5, fontweight="bold", color=MUTED)
    ax.text(x + w / 2, 3.9, body, ha="center", va="center", fontsize=11, linespacing=1.45, color=tc)
for (x, w, *_), (x2, *_) in zip(boxes[:-1], boxes[1:]):
    ax.add_patch(FancyArrowPatch((x + w + 0.5, 5), (x2 - 0.5, 5), arrowstyle="-|>", mutation_scale=15, color=MUTED, lw=1.6))
ax.text(41.5, -0.9, "green = new in this part", ha="center", va="center", fontsize=10, color=NEW)


def panel(x, y, w, h, pad, no, title, sub):
    """A titled chart area; `pad` leaves room for tick labels left of the axes."""
    fig.text(x, y + h + 0.052, f"{no}", fontsize=15, fontweight="bold", color="white", va="center", ha="center",
             bbox=dict(boxstyle="circle,pad=0.25", fc=INK, ec="none"))
    fig.text(x + 0.027, y + h + 0.052, title, fontsize=14.5, fontweight="bold", va="center")
    fig.text(x - 0.008, y + h + 0.024, sub, fontsize=10.5, color=MUTED, va="center")
    a = fig.add_axes([x + pad, y, w - pad, h])
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        a.spines[s].set_color(RULE)
    a.tick_params(length=0, labelsize=10.5)
    return a


gain = lambda v: ("+" if v > 0 else "−" if v < 0 else "") + f"{abs(v):.5f}"

# ---------------------------------------------------------------- 1. how predictable is each column
# (column, accuracy on the synthetic data, accuracy on the original data); 5-fold, from the other 20 columns
COLS = [("Customer Type", 0.992, 0.989), ("Type of Travel", 0.991, 0.989),
        ("Inflight entertainment", 0.954, 0.970), ("Class", 0.919, 0.868),
        ("Ease of Online booking", 0.918, 0.931), ("Inflight wifi service", 0.908, 0.913),
        ("Dep./Arr. time convenient", 0.756, 0.723), ("Online boarding", 0.739, 0.745),
        ("Seat comfort", 0.733, 0.692), ("Cleanliness", 0.705, 0.714),
        ("Gate location", 0.683, 0.649), ("Food and drink", 0.672, 0.685),
        ("Baggage handling", 0.649, 0.609), ("On-board service", 0.568, 0.559),
        ("Leg room service", 0.514, 0.503), ("Checkin service", 0.315, 0.303)]
a = panel(0.05, 0.085, 0.42, 0.62, 0.175, 1, "Most columns can be guessed", "accuracy of predicting a column from the other 20 (section 8)")
for i, (lab, syn, org) in enumerate(COLS):
    a.plot([0, syn], [i, i], color=RULE, lw=1.2, zorder=1)
    a.scatter([org], [i], s=70, facecolor="white", edgecolor=MUTED, linewidth=1.6, zorder=2)
    a.scatter([syn], [i], s=62, color=NEW, zorder=3)
    a.text(1.06, i, f"{syn:.0%}", va="center", fontsize=10.5, color=NEW, fontweight="bold")
a.set_yticks(range(len(COLS))); a.set_yticklabels([c[0] for c in COLS]); a.set_ylim(len(COLS) - 0.4, -2.5)
a.set_xlim(0, 1.03); a.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); a.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
a.spines["left"].set_visible(False); a.grid(axis="x", color=RULE, lw=0.6); a.set_axisbelow(True)
a.scatter([0.04], [-1.95], s=62, color=NEW)
a.text(0.085, -1.95, "this competition's data (synthetic)", va="center", fontsize=10.5, color=NEW, fontweight="bold")
a.scatter([0.04], [-1.2], s=70, facecolor="white", edgecolor=MUTED, linewidth=1.6)
a.text(0.085, -1.2, "the original data it was made from", va="center", fontsize=10.5, color=MUTED, fontweight="bold")

# ---------------------------------------------------------------- 2. which feature helps
a = panel(0.56, 0.455, 0.41, 0.25, 0.0, 2, "The expected value is what helps",
          "one XGBoost, fold 0: change in AUC per feature group (section 8)")
abl = [("13 expected values", 0.00010), ("all 30 new features", 0.00004), ("row log-likelihood only", 0.00002),
       ("16 surprise probabilities only", -0.00010)]
for i, (lab, v) in enumerate(abl):
    a.barh(i, v, height=0.36, color=SEEN if v > 0 else MUTED)
    a.text(0.000004 if v < 0 else -0.000004, i, lab, va="center", ha="left" if v < 0 else "right", fontsize=10.5)
    a.text(v + (0.000004 if v > 0 else -0.000004), i, gain(v), va="center", ha="left" if v > 0 else "right",
           fontsize=10.5, fontweight="bold", color=SEEN if v > 0 else MUTED)
a.axvline(0, color=INK, lw=1); a.set_ylim(len(abl) - 0.4, -0.6); a.set_xlim(-0.00016, 0.00016)
a.set_yticks([]); a.set_xticks([]); a.spines["left"].set_visible(False); a.spines["bottom"].set_visible(False)

# ---------------------------------------------------------------- 3. what it is worth
a = panel(0.56, 0.085, 0.41, 0.25, 0.0, 3, "+0.0001 per model, +0.00006 in the stack",
          "5-fold AUC gain from the new features (scoreboard and section 9)")
worth = [("one XGBoost", 0.00013, SEEN), ("one LightGBM", 0.00009, SEEN), ("the stack: 26 → 29 models", 0.00006, REAL),
         ("public leaderboard", 0.00002, REAL)]
for i, (lab, v, col) in enumerate(worth):
    a.barh(i, v, height=0.34, color=col)
    a.text(0, i - 0.36, lab, va="center", fontsize=10.5)
    a.text(v + 0.000003, i, gain(v), va="center", fontsize=10.5, color=col, fontweight="bold")
a.plot([0.000035, 0.000035], [1.76, 2.24], color=INK, lw=1.6)
a.text(0.000105, 2, "│ = all of Part 3, +0.00004", va="center", fontsize=10, color=MUTED)
a.text(0.000066, 3, "below what the public leaderboard can resolve", va="center", fontsize=10, color=MUTED)
a.set_ylim(len(worth) - 0.45, -0.75); a.set_xlim(0, 0.00021); a.set_yticks([]); a.set_xticks([])
a.spines["bottom"].set_visible(False)

fig.text(0.04, 0.018, "Orange = one model in cross-validation.  Blue = the stack or the leaderboard.  Green = new in Part 4.",
         fontsize=10.5, color=MUTED, va="center")
fig.savefig("part4_summary.png", facecolor="white")
plt.show()

In Part 3 we wrote that there was nothing left to find with these 21 columns. That was wrong, and this notebook is the correction. What was left was not a new column or a new model family but a different question put to the same data:

1. **The idea.** Every model so far predicts the label from the columns. Here 16 extra models each predict *one column from the other 20*, without the label, on all 1M train and test rows (section 8).
2. **What it gives one model.** The expected value of each rating "given the rest of the row" becomes 13 new features. One XGBoost goes from 0.96108 to 0.96121 and one LightGBM from 0.96107 to 0.96116.
3. **What it gives the stack.** Three such trees lift the 26-model stack of Part 3 by +0.000059, with all five folds up. Everything in Part 3 together was +0.000035, and the soft labels of Part 3, which lifted single models by more than this, added nothing to the stack. This gain survives stacking. On the leaderboard it is +0.00002, which the public leaderboard cannot tell from noise; the paired CV can.
4. **What we expected and did not find.** We built these models to measure how *surprising* a row is. The surprise features do nothing for a single model. The useful part is the plain expected value (section 8).

| step | nested CV | public LB |
|---|---|---|
| Part 1: 13 view-A models | 0.961436 | 0.96108 |
| Part 2: + DeepFM, view-B trees, `flat_anneal` RealMLP (21 models) | 0.961577 | 0.96125 |
| Part 3: + CatBoost with its own categorical statistics ×3, XGBoost from a GLM margin ×2 (26 models) | 0.961612 | 0.96127 |
| + 3 trees with auxiliary-task features (29 models, this notebook) | **0.961671** | **0.96129** |

> The LB score in the title comes from running this pipeline on our own machine. GPU training is not bit-reproducible, so a re-run can differ in the 5th decimal.

## Scoreboard: what each technique is worth

Carried forward from Parts 1 to 3. Rows marked *(new)* are from this part. Every number is a 5-fold AUC on the same split (`StratifiedKFold(5, shuffle=True, random_state=42)`); stack numbers are nested (the combiner never scores rows it was fitted on). "LB" is the public score of runs we actually submitted.

**One XGBoost, technique by technique**

| technique | CV | gain | LB |
|---|---|---|---|
| raw 21 columns | 0.958763 | | 0.95808 |
| + Flight Distance as a route ID: target encoding, value counts, route profile | 0.960362 | **+0.00160** | |
| + prediction of a model trained on the original data, as a feature (and learning rate 0.05 → 0.02) | 0.961078 | **+0.00072** | |
| + 45 Optuna trials | 0.961103 | +0.00003 | |
| + soft labels from a cross-fitted teacher | 0.961152 | +0.00005 | |
| the untuned model (0.961078) + expected value of each rating given the other 20 columns *(new)* | 0.961208 | **+0.00013** | |

**One CatBoost**

| technique | CV | gain |
|---|---|---|
| view-A features with our fold-safe target encoding | 0.960914 | |
| numeric and rating columns as categorical features, CatBoost's own ordered target statistics, no target encoding of ours | 0.961064 | **+0.00015** |
| … depth 6 – 7 and `l2_leaf_reg` 10 | 0.961126 – 0.961139 | +0.00007 |
| view-A features + soft labels (for comparison) | 0.961100 | +0.00019 |

**One RealMLP**

| technique | CV | gain | LB |
|---|---|---|---|
| public recipe (categorical twin of every numeric column, 3 epochs) | 0.960706 | | |
| + `flat_anneal` learning-rate schedule | 0.960968 | **+0.00026** | |
| public recipe + 6 epochs + engineered numeric inputs ("v4") | 0.961166 | +0.00046 (see the correction in [Part 3](https://www.kaggle.com/code/goodpjw2008/s6e10-catboost-ctr-glm-margin-xgb-lb-0-96127)) | 0.96082 |
| v4 inputs + `flat_anneal`, 3 epochs | 0.961179 | same score, half the time | |

**Other single models:** LightGBM 0.961072, with the auxiliary-task features 0.961161 (**+0.00009**) *(new)* · DeepFM 0.960746 · view-B LightGBM / XGBoost / CatBoost 0.961005 / 0.960937 / 0.960811 · GLM on 120 target-rate logits 0.958541 · XGBoost boosted from that GLM 0.960752 – 0.960808 · nearest neighbours 0.956214 · FT-Transformer 0.9592 (fold 0) · GPT-2 fine-tuned as a classifier 0.9574 (fold 0)

**The stack, member group by member group**

| stack | nested CV | gain | LB |
|---|---|---|---|
| best single model (RealMLP v4) | 0.961166 | | 0.96082 |
| 13 view-A models | 0.961436 | **+0.00027** | 0.96108 |
| + DeepFM | 0.961450 | +0.00001 | |
| + 3 view-B trees | 0.961500 | **+0.00005** | |
| + RealMLP with the `flat_anneal` schedule | 0.961577 | **+0.00008** | 0.96125 |
| + 3 CatBoost with native categorical statistics | 0.961603 | +0.00003 | |
| + 2 XGBoost boosted from a GLM margin | 0.961612 | +0.00001 | 0.96127 |
| + 3 trees with auxiliary-task features *(new)* | 0.961671 | **+0.00006** | **0.96129** |
| *for contrast:* the same kind of members with 10 folds instead of 5 | +0.00012 in CV | | 0.96107 (no gain) |

The pattern over four notebooks: two feature ideas were worth +0.0023, stacking +0.0003, a second view and a schedule +0.00014, two more tree variants +0.00004, and the auxiliary-task features +0.00006 in the stack (+0.0001 for a single tree model).

## Credits and thanks

| Whose work | What we took from it | What we changed or added |
|---|---|---|
| **The S6E9 9th-place write-up** ([link](https://www.kaggle.com/competitions/playground-series-s6e9/writeups/9th-place-solution)) | One line of its model table: "Auxiliary models: additional structure from auxiliary tasks". The write-up gives no code and does not say which tasks, so we took the direction, not a method. (The same write-up kindly cites our S6E9 notebook, which is how we came to read it.) | The task (every column from the other 20, no label, train and test together), the three kinds of feature, the ablation that shows which of them matters, and all code. |
| **thisray** ([is your 0.0001 gain real?](https://www.kaggle.com/code/thisray/s6e10-is-your-0-0001-gain-real-noise-measured)) | The resampling estimate that the public leaderboard cannot order gaps below about 0.0003, while 700k out-of-fold rows resolve about 0.00005. It is why we judge this part by paired CV and not by its +0.00002 on the leaderboard. | Unchanged. |
| **heuljax**, **megayak / 인공저능연구소**, **Arhan canli**, **Vladimir Demidov**, **Dariush Afshar**, **TJ Klein**, **the pytabkit authors** | As listed in [Part 2](https://www.kaggle.com/code/goodpjw2008/s6e10-two-view-gbdt-realmlp-stack-lb-0-96125) and [Part 3](https://www.kaggle.com/code/goodpjw2008/s6e10-catboost-ctr-glm-margin-xgb-lb-0-96127): the GLM-margin recipe, the view-B ingredients, the `flat_anneal` schedule, GPT-2 token keys, the RealMLP recipe, independent verification, the original data, RealMLP itself. | Unchanged from Parts 2 and 3. |

What is ours in this part: the auxiliary-task features and their ablation (section 8), the comparison of how predictable each column is in the original and in the synthetic data, and the correction of Part 3 (section 10).

No public prediction files or OOF libraries were used in this part, not even for measurement.

## 0. Setup

Add the competition data and the dataset `teejmahal20/airline-passenger-satisfaction` as inputs, turn the GPU on, and keep Internet on (for `pytabkit` and the GPT-2 tokenizer files).
Part 2 took 2 hours 16 minutes on a Kaggle GPU for 21 models; this notebook trains 26, so expect a little over three hours.

In [ ]:
!pip install -q pytabkit tiktoken

In [ ]:
import glob, logging, os, time, warnings
import numpy as np, pandas as pd
import torch
import torch.nn as nn
import tiktoken
import xgboost as xgb, lightgbm as lgb
from catboost import CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.preprocessing import TargetEncoder
warnings.filterwarnings("ignore")
for name in ("lightning", "lightning.pytorch", "pytorch_lightning"):   # keep the training log short
    logging.getLogger(name).setLevel(logging.ERROR)

INPUT = os.environ.get("INPUT_DIR", "/kaggle/input")
SMOKE = os.environ.get("SMOKE") == "1"          # tiny dry-run to check the code end to end
GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
N_EST, ES = (300, 50) if SMOKE else (20000, 400)
print("GPU:", GPU, "| smoke run:", SMOKE)

def find(name, hint):
    hits = sorted(p for p in glob.glob(f"{INPUT}/**/{name}", recursive=True) if hint in p)
    assert hits, f"{name} ({hint}) not found under {INPUT}"
    return hits[0]

## 1. Data and folds

The original dataset we attach has one more column (`Inflight service`) than the competition data; we drop it. (The table the generator was actually trained on never had that column, see section 12.)
Every model below uses the same `StratifiedKFold(5, shuffle=True, random_state=42)` so that out-of-fold predictions line up row for row.

In [ ]:
TARGET = "satisfaction"
FD = "Flight Distance"
RATINGS = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
           "Gate location", "Food and drink", "Online boarding", "Seat comfort", "Inflight entertainment",
           "On-board service", "Leg room service", "Baggage handling", "Checkin service", "Cleanliness"]
CATS = ["Gender", "Customer Type", "Type of Travel", "Class"]
NUMS = ["Age", FD, "Departure Delay in Minutes", "Arrival Delay in Minutes"]
FEATS = NUMS + RATINGS + CATS

tr = pd.read_csv(find("train.csv", "playground-series-s6e10"))
te = pd.read_csv(find("test.csv", "playground-series-s6e10"))
og = pd.concat([pd.read_csv(find(f, "airline-passenger-satisfaction"), index_col=0)
                for f in ("train.csv", "test.csv")], ignore_index=True)
if SMOKE:
    tr, te = tr.sample(30000, random_state=0).reset_index(drop=True), te.head(5000).copy()

tr[TARGET] = tr[TARGET].astype(int)
og[TARGET] = (og[TARGET] == "satisfied").astype(int)
og = og[FEATS + [TARGET]].drop_duplicates().reset_index(drop=True)

y, NTR = tr[TARGET].values, len(tr)
full = pd.concat([tr[FEATS], te[FEATS]], ignore_index=True)       # train rows first, then test
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR), y))
print(tr.shape, te.shape, og.shape, "| positive rate", y.mean().round(4))
print("distinct Flight Distance values:", full[FD].nunique(),
      "| rows covered by the 20 most frequent:", full[FD].value_counts().head(20).sum())

## 2. Knowledge from the original dataset

Train on the original data **only**, predict the competition rows, and keep those predictions as features.
No competition label is involved, so the features are leak-free and are computed once for train + test.

The cell also shows why this works. Where the original-data model is certain the passenger is dissatisfied, about 3.5% of the competition labels are positive anyway; where it is certain they are satisfied, about 5.5% are negative. The competition target is the original decision function plus label noise, and a model fitted on clean labels supplies detail that 700k noisy rows do not recover on their own.

Three original-data models are fitted here: XGBoost and RealMLP feed view A, LightGBM feeds view B. Giving the two views different teachers is part of keeping them apart.

In [ ]:
OG = pd.DataFrame(index=range(len(full)))

# --- XGBoost trained on the original rows
both = pd.concat([full, og[FEATS]], ignore_index=True)
for c in CATS:
    both[c] = both[c].astype("category").cat.codes
m = xgb.XGBClassifier(n_estimators=600, learning_rate=0.05, max_depth=8, subsample=0.8,
                      colsample_bytree=0.5, device=DEVICE, random_state=0)
m.fit(both.iloc[len(full):], og[TARGET])
OG["og_xgb_d8"] = m.predict_proba(both.iloc[:len(full)])[:, 1].astype(np.float32)

p = OG["og_xgb_d8"].values[:NTR]
print(f"original-data model scored against competition labels: AUC {roc_auc_score(y, p):.5f}")
zone = pd.cut(p, [0, 0.02, 0.98, 1], labels=["sure negative", "unsure", "sure positive"], include_lowest=True)
print(pd.DataFrame({"zone": zone, "y": y}).groupby("zone", observed=True)["y"]
        .agg(rows="size", positive_rate="mean").round(4))

# --- LightGBM trained on the original rows (the teacher of the view-B trees, section 6)
m = lgb.LGBMClassifier(n_estimators=1500, learning_rate=0.03, num_leaves=63, subsample=0.8, subsample_freq=1,
                       colsample_bytree=0.5, verbose=-1, n_jobs=-1)
m.fit(both.iloc[len(full):], og[TARGET])
OG["og_lgb"] = m.predict_proba(both.iloc[:len(full)])[:, 1].astype(np.float32)
print(f"og_lgb vs competition labels: AUC {roc_auc_score(y, OG['og_lgb'].values[:NTR]):.5f}")

In [ ]:
# RealMLP hyper-parameters from Vladimir Demidov's public notebook (thank you!)
REALMLP = dict(
    n_ens=8, n_epochs=3, batch_size=256, use_early_stopping=False,
    lr=0.053, wd=0.0236, sq_mom=0.988, lr_sched="lin_cos_log_15", wd_sched="cos_log_15",
    first_layer_lr_factor=0.25, embedding_size=5, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act="silu", p_drop=0.05, p_drop_sched="expm4t",
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name="gelu", plr_lr_factor=0.1151, plr_sigma=2.33,
    ls_eps=0.01, ls_eps_sched="sqrt_cos", add_front_scale=False, bias_init_mode="neg-uniform-dynamic-2",
    tfms=["one_hot", "median_center", "robust_scale", "smooth_clip", "embedding", "l2_normalize"],
)

def nn_frame(X):
    # RealMLP input. Also from that notebook: every raw numeric column gets a categorical twin,
    # so the net learns an embedding per distinct value (e.g. per route).
    # Original-model probabilities are nearly 0/1, so they go in as logits.
    X = X.copy()
    X["Arrival Delay in Minutes"] = X["Arrival Delay in Minutes"].fillna(0.0)
    for c in [c for c in X.columns if c.startswith("og_")]:
        q = X[c].clip(1e-6, 1 - 1e-6)
        X[c] = np.log(q / (1 - q)).astype(np.float32)
    cat_cols = list(CATS)
    for c in NUMS + RATINGS:
        X[c + "_cat_"] = X[c].astype(int).astype(str)
        cat_cols.append(c + "_cat_")
    for c in cat_cols:
        X[c] = X[c].astype(str).astype("category")
    return X, cat_cols

# --- RealMLP trained on the original rows (3 seeds, 8 epochs: only 130k rows)
t0 = time.time()
B, b_cats = nn_frame(pd.concat([full, og[FEATS]], ignore_index=True))
pred = np.zeros(len(full))
seeds = range(1 if SMOKE else 3)
for s in seeds:
    m = RealMLP_TD_Classifier(**{**REALMLP, "n_epochs": 8}, device=DEVICE, random_state=s,
                              verbosity=0, val_fraction=0.05)
    m.fit(B.iloc[len(full):], og[TARGET].values, cat_col_names=b_cats)
    pred += m.predict_proba(B.iloc[:len(full)])[:, 1] / len(seeds)
OG["og_realmlp"] = pred.astype(np.float32)
del B; torch.cuda.empty_cache()
print(f"og_realmlp vs competition labels: AUC {roc_auc_score(y, pred[:NTR]):.5f}  ({time.time() - t0:.0f}s)")

## 3. View A: route-ID features

Three ways of telling a model *which route* a row belongs to. Measured with one XGBoost on fold 0 (raw columns: 0.958998):

| feature block | fold-0 AUC | uses target? |
|---|---|---|
| `max_bin=4096` only | 0.959747 | |
| value counts of the numeric columns | 0.959662 | no |
| **route profile**: per-route mean of every other feature | 0.960391 | **no** |
| target encoding of Flight Distance | 0.960313 | yes (fold-safe) |
| target encodings + counts + route profile | 0.960652 | |
| … + original-model prediction | **0.961328** | |

The route profile reaches the level of target encoding without touching the label: what the model needs is a dense description of the route, not target statistics as such.
The original-model prediction adds +0.0002 on raw columns but **+0.0008** next to the route profile.

Target encoding is done inside each CV fold with an inner 5-fold, so a training row never sees its own label.
Crossing Flight Distance with *every* other feature, or encoding rating combinations, overfits and is left out.

In [ ]:
# value counts over train+test
CNT = pd.DataFrame({f"cnt_{c}": full[c].map(full[c].value_counts()).fillna(0).astype(np.float32) for c in NUMS})

# route profile: mean of every other feature per Flight Distance value (no target involved)
coded = full.copy()
for c in CATS:
    coded[c] = coded[c].astype("category").cat.codes
grp = coded.groupby(FD)
FDAGG = pd.DataFrame({f"fdm_{c}": grp[c].transform("mean").astype(np.float32) for c in FEATS if c != FD})
FDAGG["fd_cnt"] = grp["Age"].transform("size").astype(np.float32)

# target-encoding keys
def combine_keys(cols):
    code = np.zeros(len(full), dtype=np.int64)
    for c in cols:
        k = pd.factorize(full[c], use_na_sentinel=False)[0]
        code = code * (k.max() + 1) + k
    return pd.factorize(code)[0]

TE_KEYS = [[FD], ["Age"], [FD, "Age"], [FD, "Class"], [FD, "Type of Travel"],
           ["Age", "Class", "Type of Travel", "Customer Type"]]
KEY = {"te_" + "|".join(c[:6] for c in cols): combine_keys(cols) for cols in TE_KEYS}

def target_encode(k_fit, y_fit, k_apply, smooth=20.0):
    prior = y_fit.mean()
    s = pd.DataFrame({"k": k_fit, "y": y_fit}).groupby("k")["y"].agg(["sum", "count"])
    enc = (s["sum"] + prior * smooth) / (s["count"] + smooth)
    return pd.Series(k_apply).map(enc).fillna(prior).values.astype(np.float32)

def add_te(Xa, ya, others, keys=None):
    # Training rows: inner 5-fold encodings. Validation/test rows: encoded with all training rows of the fold.
    # Rows are identified by their index, which is their position in `full`.
    Xa, others = Xa.copy(), [o.copy() for o in others]
    for name, codes in (KEY if keys is None else keys).items():
        ka, col = codes[Xa.index.values], np.zeros(len(Xa), dtype=np.float32)
        for i, j in KFold(5, shuffle=True, random_state=0).split(ka):
            col[j] = target_encode(ka[i], ya[i], ka[j])
        Xa[name] = col
        for o in others:
            o[name] = target_encode(ka, ya, codes[o.index.values])
    return Xa, others

def static_frame(og_cols=(), fdagg=False, cnt=False):
    X = full.copy()
    for c in CATS:
        X[c] = X[c].astype("category")
    if fdagg:
        X = pd.concat([X, FDAGG], axis=1)
    for c in og_cols:
        X[c] = OG[c].values
    if cnt:
        X = pd.concat([X, CNT], axis=1)
    return X

OOF, TEST = {}, {}
def run_cv(name, fit, X, te=True):
    Xtr, Xte = X.iloc[:NTR], X.iloc[NTR:]
    oof, tst, t0 = np.zeros(NTR), np.zeros(len(Xte)), time.time()
    for a, b in FOLDS:
        Xa, Xb, Xt = Xtr.iloc[a], Xtr.iloc[b], Xte
        if te:
            Xa, (Xb, Xt) = add_te(Xa, y[a], [Xb, Xt])
        predict = fit(Xa, y[a], Xb, y[b])
        oof[b] = predict(Xb)
        tst += predict(Xt) / len(FOLDS)
        del predict; torch.cuda.empty_cache()
    OOF[name], TEST[name] = oof, tst
    print(f"{name:16s} OOF AUC {roc_auc_score(y, oof):.6f}   ({time.time() - t0:.0f}s)")

## 4. View A: gradient-boosted trees

All tree models share one feature set: raw columns + route profile + original-model prediction(s) + value counts + fold-safe target encodings.
An Optuna search over XGBoost (45 trials, scored on two folds) moved the score by only +0.00007 — the features decide, the hyper-parameters barely matter. The four XGBoost variants are in the stack for diversity rather than because one is clearly better.

In [ ]:
def fit_xgb(seed=42, **over):
    P = dict(n_estimators=N_EST, learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.5,
             min_child_weight=5, reg_lambda=1.0, max_bin=256, tree_method="hist", device=DEVICE,
             eval_metric="auc", early_stopping_rounds=ES, enable_categorical=True, max_cat_to_onehot=1)
    P.update(over)
    def fit(Xa, ya, Xb, yb):
        m = xgb.XGBClassifier(**P, random_state=seed).fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        return lambda X: m.predict_proba(X)[:, 1]
    return fit

def fit_lgb(Xa, ya, Xb, yb):
    m = lgb.LGBMClassifier(n_estimators=N_EST, learning_rate=0.02, num_leaves=127, min_child_samples=50,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                           max_bin=255, n_jobs=-1, verbose=-1, random_state=42)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric="auc", callbacks=[lgb.early_stopping(ES, verbose=False)])
    return lambda X: m.predict_proba(X)[:, 1]

def fit_cat(Xa, ya, Xb, yb):
    m = CatBoostClassifier(iterations=N_EST, learning_rate=0.05, depth=8, l2_leaf_reg=3.0, border_count=254,
                           task_type="GPU" if GPU else "CPU", eval_metric="Logloss", early_stopping_rounds=ES,
                           random_seed=42, cat_features=CATS, verbose=0, allow_writing_files=False)
    m.fit(Xa, ya, eval_set=(Xb, yb))
    return lambda X: m.predict_proba(X)[:, 1]

X_og1 = static_frame(og_cols=["og_xgb_d8"], fdagg=True, cnt=True)
X_og2 = static_frame(og_cols=["og_xgb_d8", "og_realmlp"], fdagg=True, cnt=True)

GBDT_MODELS = [
    ("xgb_v3",       fit_xgb(),                                                    X_og1),
    ("xgb_v3_s1",    fit_xgb(seed=1, colsample_bytree=0.4, max_depth=7),           X_og2),
    ("xgb_v3_d10",   fit_xgb(seed=2, max_depth=10, min_child_weight=20),           X_og1),
    ("xgb_v4_tuned", fit_xgb(max_depth=9, min_child_weight=55, subsample=0.92, colsample_bytree=0.8,
                             colsample_bynode=0.5, reg_lambda=3.55, reg_alpha=0.012, gamma=0.26), X_og2),
    ("lgb_v3",       fit_lgb,                                                      X_og1),
    ("cat_v3",       fit_cat,                                                      X_og1),
]
for name, fit, X in GBDT_MODELS:
    run_cv(name, fit, X)

## 5. RealMLP

Starting from the public recipe (fold-0 AUC 0.960900), one change at a time:

| change | fold-0 AUC |
|---|---|
| 2 / 4 / **6** / 10 / 16 epochs | 0.960604 / 0.961088 / **0.961167** / 0.961188 / 0.961212 |
| embedding size 8 or 16, dropout 0.1, lr 0.03 or 0.08, batch 512, label smoothing 0 or 0.03 | 0.96087 – 0.96096 (no effect) |
| + target encodings + route profile + original-model logits (3 epochs) | 0.961187 |
| … and 6 epochs | 0.961344 |
| **… and value counts, 6 epochs** | **0.961413** |
| … but 10 epochs, or hidden sizes 1024-512-256 | 0.961271, 0.961317 (no better) |

With the engineered inputs RealMLP becomes the best single model, ahead of XGBoost. It also becomes more similar to the trees, so the stack keeps the plain public-recipe members too: they are weaker alone but add diversity.

**New in this notebook: the learning-rate schedule.** Same public recipe, raw inputs, fold 0:

| `lr_sched` | epochs | fold-0 AUC |
|---|---|---|
| `lin_cos_log_15` (the recipe as published) | 3 | 0.960900 |
| **`flat_anneal`** | 3 | **0.961136** |
| `flat_anneal` | 5 | 0.961170 |
| `flat_cos` | 3 | 0.961133 |
| `constant` | 2 | 0.960624 |

Over five folds the `flat_anneal` models score 0.96095 – 0.96098 against 0.96071 – 0.96077 for the published schedule. With the engineered inputs the schedule changes little (fold 0: 0.961427 at 3 epochs against 0.961413 at 6 with the old schedule), but it halves the training time.

In the stack, one raw-input `flat_anneal` model is worth +0.00008, more than the three view-B trees together. Further seeds of the same setting add nothing to the out-of-fold score; they are kept to steady the test predictions.

In [ ]:
def fit_realmlp(cat_cols, seed=42, **over):
    def fit(Xa, ya, Xb, yb):
        m = RealMLP_TD_Classifier(**{**REALMLP, **over}, device=DEVICE, random_state=seed,
                                  val_metric_name="1-auc_ovr", verbosity=0)
        m.fit(Xa, ya, X_val=Xb, y_val=yb, cat_col_names=cat_cols)
        return lambda X: m.predict_proba(X)[:, 1]
    return fit

N_raw, cats = nn_frame(static_frame())                                              # public recipe
N_og,  _    = nn_frame(static_frame(og_cols=["og_xgb_d8", "og_realmlp"]))           # + original-model logits
N_v4,  _    = nn_frame(static_frame(og_cols=["og_xgb_d8", "og_realmlp"], fdagg=True, cnt=True))

NN_MODELS = [  # name, fit, frame, add target encodings?
    ("realmlp_pub",    fit_realmlp(cats, seed=42),                        N_raw, False),
    ("realmlp_pub_s1", fit_realmlp(cats, seed=1),                         N_raw, False),
    ("realmlp_pub_s2", fit_realmlp(cats, seed=2),                         N_raw, False),
    ("realmlp_og",     fit_realmlp(cats, seed=42),                        N_og,  False),
    ("realmlp_v4",     fit_realmlp(cats, seed=42, n_epochs=6),            N_v4,  True),
    ("realmlp_v4_s1",  fit_realmlp(cats, seed=1, n_epochs=6),             N_v4,  True),
    ("realmlp_v4_s2",  fit_realmlp(cats, seed=2, n_epochs=5, p_drop=0.1), N_v4,  True),
    # new: the flat_anneal schedule
    ("realmlp_fa",      fit_realmlp(cats, seed=42,  lr_sched="flat_anneal", n_epochs=4), N_raw, False),
    ("realmlp_fa_s7",   fit_realmlp(cats, seed=7,   lr_sched="flat_anneal", n_epochs=4), N_raw, False),
    ("realmlp_fa_s123", fit_realmlp(cats, seed=123, lr_sched="flat_anneal", n_epochs=5), N_raw, False),
    ("realmlp_v4fa",    fit_realmlp(cats, seed=11,  lr_sched="flat_anneal", n_epochs=3), N_v4,  True),
]
for name, fit, X, te_flag in NN_MODELS:
    run_cv(name, fit, X, te=te_flag)

## 5b. DeepFM: a third model family (honest result: almost no gain)

The signal that lifted every model is "which exact value does a column take" and how such values interact, which is what a factorization machine models:

`logit = bias + Σ w[value] + Σ ⟨v[value_j], v[value_k]⟩ + MLP(embeddings, numeric features)`

Every one of the 21 columns is a field. The whole fold sits on the GPU, so an epoch takes about a second. It overfits after the second epoch, hence 3 epochs and 8 nets per fold.

It reaches 0.96075 alone and adds +0.00002 to the stack. Its rank correlation with RealMLP is 0.98 and with XGBoost 0.974, about the same as RealMLP with XGBoost, so it is another reader of the same information rather than a new source. It stays in because it was part of the submitted stack.

In [ ]:
class DeepFM(nn.Module):
    def __init__(self, sizes, n_num, d=16, hidden=(512, 256, 128), drop=0.1):
        super().__init__()
        self.offsets = torch.tensor(np.concatenate([[0], np.cumsum(sizes)[:-1]]), device=DEVICE)
        n = int(sum(sizes))
        self.emb, self.lin = nn.Embedding(n, d), nn.Embedding(n, 1)
        nn.init.normal_(self.emb.weight, std=0.01)       # values never seen in training stay neutral
        nn.init.zeros_(self.lin.weight)
        self.num_bn = nn.BatchNorm1d(n_num)
        layers, i = [], len(sizes) * d + n_num
        for h in hidden:
            layers += [nn.Linear(i, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(drop)]
            i = h
        self.mlp = nn.Sequential(*layers, nn.Linear(i, 1))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, xc, xn):
        idx = xc + self.offsets
        e = self.emb(idx)
        out = self.bias + self.lin(idx).sum(1).squeeze(-1)
        out = out + 0.5 * (e.sum(1).pow(2) - e.pow(2).sum(1)).sum(1)      # all pairwise inner products
        return out + self.mlp(torch.cat([e.flatten(1), self.num_bn(xn)], 1)).squeeze(-1)

def deepfm_net(Ca, Na, ya, evals, sizes, seed, epochs=3, lr=1e-3, bs=2048):
    torch.manual_seed(seed)
    m = DeepFM(sizes, Na.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * (len(ya) // bs), pct_start=0.2)
    lossf = nn.BCEWithLogitsLoss()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(ya), device=DEVICE)
        for i in range(len(ya) // bs):
            ix = perm[i * bs:(i + 1) * bs]
            opt.zero_grad(set_to_none=True)
            lossf(m(Ca[ix], Na[ix]), ya[ix]).backward()
            opt.step(); sched.step()
    m.eval()
    with torch.no_grad():
        return [torch.cat([m(C[i:i + 65536], N[i:i + 65536]) for i in range(0, len(C), 65536)]).cpu().numpy()
                for C, N in evals]

def run_deepfm(name, n_ens=8):
    t0 = time.time()
    num = static_frame(og_cols=["og_xgb_d8", "og_realmlp"], fdagg=True, cnt=True).drop(columns=CATS)
    num["Arrival Delay in Minutes"] = num["Arrival Delay in Minutes"].fillna(0.0)
    for c in num.columns:
        if c.startswith("og_"):
            q = num[c].clip(1e-6, 1 - 1e-6); num[c] = np.log(q / (1 - q))
        elif c.startswith("cnt_") or c in ("fd_cnt", FD, "Departure Delay in Minutes", "Arrival Delay in Minutes"):
            num[c] = np.log1p(num[c])
    codes = [pd.factorize(full[c].astype(object).where(full[c].notna(), "nan"))[0] for c in FEATS]
    sizes = [int(k.max()) + 1 for k in codes]
    C = torch.tensor(np.stack(codes, 1), device=DEVICE, dtype=torch.long)
    oof, tst = np.zeros(NTR), np.zeros(len(full) - NTR)
    for a, b in FOLDS:
        Xa, (Xb, Xt) = add_te(num.iloc[:NTR].iloc[a], y[a], [num.iloc[:NTR].iloc[b], num.iloc[NTR:]])
        mu, sd = Xa.values.mean(0), Xa.values.std(0) + 1e-6
        T = lambda X: torch.tensor(np.clip((X.values - mu) / sd, -6, 6), device=DEVICE, dtype=torch.float32)
        Na, Nb, Nt = T(Xa), T(Xb), T(Xt)
        ya = torch.tensor(y[a], device=DEVICE, dtype=torch.float32)
        ia, ib = torch.tensor(a, device=DEVICE), torch.tensor(b, device=DEVICE)
        for s in range(n_ens):
            pb, pt = deepfm_net(C[ia], Na, ya, [(C[ib], Nb), (C[NTR:], Nt)], sizes, seed=s)
            oof[b] += pb / n_ens
            tst += pt / n_ens / len(FOLDS)
    sig = lambda z: 1 / (1 + np.exp(-z))
    OOF[name], TEST[name] = sig(oof), sig(tst)
    print(f"{name:16s} OOF AUC {roc_auc_score(y, oof):.6f}   ({time.time() - t0:.0f}s)")

run_deepfm("deepfm_v1", n_ens=2 if SMOKE else 8)

## 6. View B: a second description of the same rows

A stack gains from members that are good **and** different. The measurement that started this section:

| stack (same 5 folds, nested CV) | AUC |
|---|---|
| our 14 view-A models | 0.961450 |
| a public library of 6 models, alone | 0.961413 |
| ours + that library | 0.961571 |
| ours + **our own** view-B trees (below) | 0.961500 |
| … then adding the public library's three tree models on top | 0.961510 |

So an independently built pipeline adds +0.00012 to ours, and once our own view B is in, the public trees add only +0.00001 more: the tree-side diversity is reproduced. (The rest came from RealMLP, see section 5.)

A first attempt at a second view failed in an instructive way: without any original-data model score the trees reached only 0.9605 – 0.9608 and added +0.00002. Different is not enough; the members have to be strong as well.

| | view A (sections 3 – 5) | view B (this section) |
|---|---|---|
| the route | route profile: per-route mean of every other feature | not used |
| original-data knowledge | XGBoost and RealMLP scores | LightGBM score, plus the original target mean for every value of every column |
| target encoding | our own, fixed smoothing 20 | scikit-learn `TargetEncoder`, automatic smoothing |
| encoded keys | distance, age, distance × age / class / travel type | distance, distance // 10, distance % 1000, first and last GPT-2 token of the distance, age × class, distance × class, age × travel type |
| extras | value counts | value counts, digits, GPT-2 token ids of the four numeric columns |

On top of view A, these same lookup, digit and token features add nothing to a single model (fold 0: 0.961383 → 0.961340 … 0.961436). Their value is as a separate view.

In [ ]:
enc = tiktoken.get_encoding("gpt2")

VB = full.copy()
for c in CATS:
    VB[c] = VB[c].astype("category").cat.codes
q = OG["og_lgb"].clip(1e-6, 1 - 1e-6)
VB["og_lgb"] = np.log(q / (1 - q)).astype(np.float32)
# digits: a tree can isolate a single distance or age with a few splits
VB["fd_mod10"], VB["fd_mod100"], VB["fd_div10"], VB["age_mod10"] = full[FD] % 10, full[FD] % 100, full[FD] // 10, full["Age"] % 10
# original-data satisfaction rate for the value this row has in each column
for c in FEATS:
    VB[f"om_{c}"] = full[c].map(og.groupby(c)[TARGET].mean()).astype(np.float32)
# GPT-2 tokens of the numbers: 694 is written " 6" + "94"
for c in NUMS:
    col = full[c].fillna(-1).astype(int)
    T = {v: enc.encode(" " + str(v)) for v in col.unique()}
    VB[f"tk1_{c}"] = col.map({v: t[0] for v, t in T.items()}).values
    VB[f"tkl_{c}"] = col.map({v: t[-1] for v, t in T.items()}).values
    VB[f"tkn_{c}"] = col.map({v: len(t) for v, t in T.items()}).values
VB = pd.concat([VB, CNT], axis=1)

fd, age = full[FD], full["Age"]
KB = pd.DataFrame({
    "fd": fd.astype(str), "fd10": (fd // 10).astype(str), "fd_m1000": (fd % 1000).astype(str),
    "fd_t1": VB["tk1_" + FD].astype(str), "fd_tl": VB["tkl_" + FD].astype(str),
    "age_cls": age.astype(str) + full["Class"], "fd_cls": fd.astype(str) + full["Class"],
    "age_tt": age.astype(str) + full["Type of Travel"]})
ES_B = 50 if SMOKE else 300

def fit_lgb_b(Xa, ya, Xb, yb):
    m = lgb.LGBMClassifier(n_estimators=N_EST, learning_rate=0.02, num_leaves=127, min_child_samples=100,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                           n_jobs=-1, verbose=-1, random_state=42)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric="auc", callbacks=[lgb.early_stopping(ES_B, verbose=False)])
    return lambda X: m.predict_proba(X)[:, 1]

def fit_xgb_b(Xa, ya, Xb, yb):
    m = xgb.XGBClassifier(n_estimators=N_EST, learning_rate=0.02, max_depth=6, min_child_weight=5, subsample=0.8,
                          colsample_bytree=0.5, reg_lambda=2.0, max_bin=1024, tree_method="hist", device=DEVICE,
                          eval_metric="auc", early_stopping_rounds=ES_B, random_state=42)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return lambda X: m.predict_proba(X)[:, 1]

def fit_cat_b(Xa, ya, Xb, yb):
    m = CatBoostClassifier(iterations=N_EST, learning_rate=0.06, depth=7, l2_leaf_reg=3.0, border_count=254,
                           task_type="GPU" if GPU else "CPU", eval_metric="Logloss", early_stopping_rounds=ES_B,
                           random_seed=42, verbose=0, allow_writing_files=False)
    m.fit(Xa, ya, eval_set=(Xb, yb))
    return lambda X: m.predict_proba(X)[:, 1]

def run_cv_b(name, fit):
    Xtr, Xte, Ktr, Kte = VB.iloc[:NTR], VB.iloc[NTR:], KB.iloc[:NTR], KB.iloc[NTR:]
    cols = ["te_" + c for c in KB.columns]
    oof, tst, t0 = np.zeros(NTR), np.zeros(len(Xte)), time.time()
    for a, b in FOLDS:
        # fit_transform cross-fits inside the training rows, so no row sees its own label
        e = TargetEncoder(target_type="binary", smooth="auto", cv=5, shuffle=True, random_state=0)
        Ta = pd.DataFrame(e.fit_transform(Ktr.iloc[a], y[a]), columns=cols, index=Xtr.index[a])
        Tb = pd.DataFrame(e.transform(Ktr.iloc[b]), columns=cols, index=Xtr.index[b])
        Tt = pd.DataFrame(e.transform(Kte), columns=cols, index=Xte.index)
        predict = fit(pd.concat([Xtr.iloc[a], Ta], axis=1), y[a], pd.concat([Xtr.iloc[b], Tb], axis=1), y[b])
        oof[b] = predict(pd.concat([Xtr.iloc[b], Tb], axis=1))
        tst += predict(pd.concat([Xte, Tt], axis=1)) / len(FOLDS)
    OOF[name], TEST[name] = oof, tst
    print(f"{name:16s} OOF AUC {roc_auc_score(y, oof):.6f}   ({time.time() - t0:.0f}s)")

for name, fit in [("xgb_vb", fit_xgb_b), ("cat_vb", fit_cat_b), ("lgb_vb", fit_lgb_b)]:
    run_cv_b(name, fit)

## 7. From Part 3: two more kinds of tree model

After Part 2 we trained about twenty candidate members of different kinds and added each one to the 21-model stack. Only two families had a positive, repeatable contribution, and both are small.

**CatBoost with its own categorical statistics.** Our tree models get the route through target encodings that we compute fold by fold. CatBoost can do that itself: hand it the numeric and rating columns as *categorical* features and it builds ordered target statistics, and combinations of them, during training. We drop our own target encoding for these members so the two mechanisms do not overlap.

| CatBoost variant (fold 0) | AUC |
|---|---|
| native categorical statistics, depth 8 | 0.961259 |
| … `l2_leaf_reg` 10 | **0.961321** |
| … learning rate 0.03 | 0.961297 |
| … depth 10 | 0.961166 |
| … plus our target encoding as well | 0.961173 |
| … without the route profile | 0.961132 |

Three of them (different depth, regularisation and seed) add +0.000026 to the stack. Each alone is the best CatBoost we have (0.96106 – 0.96114 against 0.96091 with our encoding).

**XGBoost boosted from a GLM margin.** In S6E9 the late breakthrough was a logistic model on target-rate features, used as the starting point of shallow boosted trees. Here the logistic model gets 120 fold-safe target-rate logits (every single column with ratings as categories, distance and age buckets, all pairs of the 13 strongest columns, the travel × class × loyalty segment crossed with each rating) plus the original-model logits.

| member | CV | rank correlation with our XGBoost |
|---|---|---|
| the GLM alone | 0.958541 | 0.94 |
| XGBoost depth 4 boosted from the GLM | 0.960752 | 0.979 |
| XGBoost depth 6 boosted from the GLM | 0.960808 | |
| plain XGBoost, same features (for reference) | 0.961078 | 1 |

The GLM is the most *different* model we built and still adds nothing (−0.000005): different is not enough when the member is 0.003 behind. The residual trees are weaker than plain XGBoost, because this target is full of interactions an additive start does not help with, but they add +0.000018 to the stack. In S6E9 the same recipe was the strongest single model; it does not transfer.

In [ ]:
# ---- CatBoost with native categorical statistics (no target encoding of ours)
CTR = static_frame(og_cols=["og_xgb_d8", "og_realmlp"], fdagg=True, cnt=True)
for c in NUMS:
    CTR[f"cat_{c}"] = full[c].fillna(-1).astype(int)
CTR_CATS = CATS + [f"cat_{c}" for c in NUMS] + RATINGS
for c in CTR_CATS:
    CTR[c] = CTR[c].astype(str).astype("category")

def fit_cat_ctr(seed=42, **over):
    P = dict(iterations=N_EST, learning_rate=0.05, depth=8, l2_leaf_reg=3.0, border_count=254,
             task_type="GPU" if GPU else "CPU", eval_metric="Logloss", early_stopping_rounds=ES,
             verbose=0, allow_writing_files=False)
    P.update(over)
    def fit(Xa, ya, Xb, yb):
        m = CatBoostClassifier(**P, random_seed=seed, cat_features=CTR_CATS)
        m.fit(Xa, ya, eval_set=(Xb, yb))
        return lambda X: m.predict_proba(X)[:, 1]
    return fit

for name, fit in [("cat_ctr",  fit_cat_ctr(seed=42)),
                  ("cat_ctr3", fit_cat_ctr(seed=11, depth=6, l2_leaf_reg=10)),
                  ("cat_ctr5", fit_cat_ctr(seed=21, depth=7, l2_leaf_reg=10, learning_rate=0.04))]:
    run_cv(name, fit, CTR, te=False)

In [ ]:
# ---- GLM on target-rate logits, then XGBoost boosted from its margin
import itertools
from sklearn.preprocessing import StandardScaler

STRONG = ["Online boarding", "Inflight wifi service", "Class", "Type of Travel", "Customer Type",
          "Inflight entertainment", "Seat comfort", "Leg room service", "On-board service",
          "Baggage handling", "Checkin service", "Cleanliness", "Ease of Online booking"]
aux = full.assign(_fd10=full[FD] // 10, _fd100=full[FD] // 100, _age5=full["Age"] // 5,
                  _dep=np.minimum(full["Departure Delay in Minutes"], 60) // 5,
                  _arr=np.minimum(full["Arrival Delay in Minutes"].fillna(-5), 60) // 5)

def key_of(cols):
    c_ = np.zeros(len(aux), dtype=np.int64)
    for c in cols:
        k = pd.factorize(aux[c], use_na_sentinel=False)[0]
        c_ = c_ * (k.max() + 1) + k
    return pd.factorize(c_)[0]

SEG = ["Type of Travel", "Class", "Customer Type"]
glm_keys = ([[c] for c in FEATS] + [["_fd10"], ["_fd100"], ["_age5"], ["_dep"], ["_arr"]]
            + [list(p) for p in itertools.combinations(STRONG, 2)]
            + [SEG] + [SEG + [r] for r in RATINGS] + [["_age5"] + SEG, ["_fd100"] + SEG])
GKEY = {"g%03d_" % i + "|".join(c[:5] for c in k): key_of(k) for i, k in enumerate(glm_keys)}
print(len(GKEY), "target-rate keys")

lgt4 = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
GBASE = pd.DataFrame({"og_xgb": lgt4(OG["og_xgb_d8"].values), "og_mlp": lgt4(OG["og_realmlp"].values)})
ES_G = 50 if SMOKE else 300
sig = lambda z: 1 / (1 + np.exp(-z))

t0 = time.time()
glm_oof, glm_tst = np.zeros(NTR), np.zeros(len(full) - NTR)
res = {d: [np.zeros(NTR), np.zeros(len(full) - NTR)] for d in (4, 6)}
for a, b in FOLDS:
    Ga, (Gb, Gt) = add_te(GBASE.iloc[:NTR].iloc[a], y[a], [GBASE.iloc[:NTR].iloc[b], GBASE.iloc[NTR:]], keys=GKEY)
    for G in (Ga, Gb, Gt):
        for c in GKEY:
            G[c] = lgt4(G[c].values)
    sc = StandardScaler().fit(Ga.values)
    glm = LogisticRegression(C=0.05, max_iter=400).fit(sc.transform(Ga.values), y[a])
    ma, mb, mt = (glm.decision_function(sc.transform(G.values)) for G in (Ga, Gb, Gt))
    glm_oof[b], glm_tst = mb, glm_tst + mt / len(FOLDS)
    # residual trees: start from the GLM logit, learn what it misses
    Xa, (Xb, Xt) = add_te(X_og2.iloc[:NTR].iloc[a], y[a], [X_og2.iloc[:NTR].iloc[b], X_og2.iloc[NTR:]])
    for depth in res:
        m = xgb.XGBClassifier(n_estimators=N_EST, learning_rate=0.03, max_depth=depth, subsample=0.8,
                              colsample_bytree=0.5, min_child_weight=20, reg_lambda=5.0, tree_method="hist",
                              device=DEVICE, eval_metric="auc", early_stopping_rounds=ES_G,
                              enable_categorical=True, max_cat_to_onehot=1, random_state=42)
        m.fit(Xa, y[a], base_margin=ma, eval_set=[(Xb, y[b])], base_margin_eval_set=[mb], verbose=False)
        res[depth][0][b] = m.predict(Xb, base_margin=mb, output_margin=True)
        res[depth][1] += m.predict(Xt, base_margin=mt, output_margin=True) / len(FOLDS)
print(f"GLM alone        OOF AUC {roc_auc_score(y, glm_oof):.6f}   (not used in the stack)")
for depth, name in ((4, "glm_xgb"), (6, "glm6_xgb")):
    OOF[name], TEST[name] = sig(res[depth][0]), sig(res[depth][1])
    print(f"{name:16s} OOF AUC {roc_auc_score(y, OOF[name]):.6f}")
print(f"({time.time() - t0:.0f}s)")

## 8. New in Part 4: auxiliary-task features

Every model above answers one question: given the 21 columns, is the passenger satisfied? The table can answer 16 other questions that need no label at all: **given 20 of the columns, what is the 21st?**

For each of the 13 rating columns and for `Class`, `Type of Travel` and `Customer Type`:

1. Take all 1M rows, train and test together. No label is involved, so the test rows may take part.
2. Train a multi-class XGBoost that predicts the column from the other 20 columns, in 5 folds, so every row is predicted by a model that did not see it.
3. Keep three things per row:
   - `aux_ev_<column>`: the **expected value** of the rating, Σ k · P(k). "What this passenger would be expected to answer, given everything else they answered."
   - `aux_p_<column>`: the probability the model gives to the value the row **actually** has. Low means the value is surprising.
   - `aux_sum_logp`: the sum of log `aux_p` over the 16 columns, a pseudo-log-likelihood of the whole row.

Because no label is used, nothing here has to be recomputed inside the CV folds of the label models: it is computed once, like the value counts and the route profile of section 3.

The cost is 80 XGBoost fits of 300 rounds (16 columns × 5 folds).

In [ ]:
AUX_TARGETS = RATINGS + ["Class", "Type of Travel", "Customer Type"]
AUX = pd.DataFrame(index=range(len(full)))
t0 = time.time()
for c in AUX_TARGETS:
    yc = pd.factorize(coded[c], sort=True)[0]                 # `coded` = `full` with integer category codes
    k = int(yc.max()) + 1
    others = [f for f in FEATS if f != c]
    P = np.zeros((len(full), k), dtype=np.float32)
    for a, b in KFold(5, shuffle=True, random_state=0).split(coded):
        m = xgb.XGBClassifier(n_estimators=30 if SMOKE else 300, learning_rate=0.1, max_depth=6, subsample=0.8,
                              colsample_bytree=0.7, tree_method="hist", device=DEVICE, random_state=0,
                              objective="multi:softprob" if k > 2 else "binary:logistic")
        m.fit(coded.iloc[a][others], yc[a])
        P[b] = m.predict_proba(coded.iloc[b][others])
    AUX[f"aux_p_{c}"] = P[np.arange(len(full)), yc]
    if c in RATINGS:
        AUX[f"aux_ev_{c}"] = P @ np.sort(coded[c].unique()).astype(np.float32)
    print(f"{c:36s} {k} values | accuracy {(P.argmax(1) == yc).mean():.3f} | "
          f"mean probability of the actual value {AUX[f'aux_p_{c}'].mean():.3f}   ({time.time() - t0:.0f}s)")
AUX["aux_sum_logp"] = np.log(np.clip(AUX[[c for c in AUX if c.startswith("aux_p_")]].values, 1e-6, 1)).sum(1)
AUX = AUX.astype(np.float32)
EV_COLS = [c for c in AUX if c.startswith("aux_ev_")]
print(AUX.shape)

### What the column predictors say

Accuracy of predicting each column from the other 20. The first number is what the cell above prints. For comparison we ran the same thing on the 130k rows of the original dataset (5-fold), and we also took predictors fitted on the original rows only and applied them to the competition rows.

| column | values | competition data, 5-fold | original data, 5-fold | fitted on original, applied to competition rows |
|---|---|---|---|---|
| Customer Type | 2 | 99.2% | 98.9% | 99.0% |
| Type of Travel | 2 | 99.1% | 98.9% | 99.0% |
| Inflight entertainment | 6 | 95.4% | 97.0% | 93.9% |
| Class | 3 | 91.9% | 86.8% | 91.6% |
| Ease of Online booking | 6 | 91.8% | 93.1% | 90.8% |
| Inflight wifi service | 6 | 90.8% | 91.3% | 90.0% |
| Departure/Arrival time convenient | 6 | 75.6% | 72.3% | 74.5% |
| Online boarding | 6 | 73.9% | 74.5% | 73.3% |
| Seat comfort | 6 | 73.3% | 69.2% | 72.3% |
| Cleanliness | 6 | 70.5% | 71.4% | 70.1% |
| Gate location | 6 | 68.3% | 64.9% | 67.9% |
| Food and drink | 6 | 67.2% | 68.5% | 66.3% |
| Baggage handling | 5 | 64.9% | 60.9% | 64.1% |
| On-board service | 6 | 56.8% | 55.9% | 55.9% |
| Leg room service | 6 | 51.4% | 50.3% | 50.9% |
| Checkin service | 6 | 31.5% | 30.3% | 30.8% |

Reading it:

- **Most of the row is redundant.** `Customer Type` and `Type of Travel` are fixed by the other columns 99% of the time, `Inflight entertainment` 95%, and six columns are above 90%. Only `Checkin service` is close to unpredictable.
- **This is a property of the original data, not something the generator invented.** The 130k real rows are just as predictable: the two accuracy columns differ by about one point on average. The competition data is slightly *more* predictable, most of all for `Class` (91.9% against 86.8%).
- **The generator kept that structure.** Predictors that never saw a competition row lose only 0.1 to 1.5 points on the competition rows. Whatever the expected values measure, the original data already contains it.

### Which of the three kinds of feature helps

One XGBoost with the view-A features of section 4, fold 0, with the new features added. Seeds 42 / 1 / 2 where we ran more than one.

| features added | fold-0 AUC | mean | change |
|---|---|---|---|
| none | 0.961383 / 0.961408 / 0.961390 | 0.961394 | |
| the 13 expected values | 0.961450 / 0.961545 | 0.961498 | **+0.00010** |
| all 30 (expected values, probabilities of the actual value, row log-likelihood) | 0.961445 / 0.961482 / 0.961369 | 0.961432 | +0.00004 |
| the 16 probabilities of the actual value only | 0.961284 | | −0.00010 |
| the row log-likelihood only | 0.961404 | | +0.00002 |

The baseline moves by 0.00001 between seeds and the models with the new features by about 0.00005, so only the first and the fourth row are clearly different from zero.

### Five folds, and what the stack makes of it

| member | features | 5-fold CV | added alone to the 21-model stack of Part 2 (0.961577) |
|---|---|---|---|
| `xgb_auxev` | view A + 13 expected values | **0.961208** | +0.000034 |
| `xgb_auxall` | view A + all 30, another seed | 0.961189 | **+0.000062** |
| `lgb_auxall` | LightGBM, view A + all 30 | 0.961161 | +0.000048 |
| all three, on the Part 2 stack | | | +0.000070 |
| **all three, on the 26-model stack of Part 3 (0.961612)** | | | **+0.000059** → 0.961671 |

Per fold, the last line is +0.000043, +0.000056, +0.000009, +0.000099, +0.000091: all five folds up, standard error 0.000016. The test predictions of the old and the new stack have a rank correlation of 0.9981, against 0.9994 between the stacks of Part 2 and Part 3, so the new members change the ranking more than the Part 3 members did.

Reading it:

- **The expected value is the useful part.** It lifts XGBoost from 0.96108 to 0.96121 and LightGBM from 0.96107 to 0.96116. We had built these models for the other two features, the "surprise" of a row, and those do nothing for a single model.
- **A member's own score is not its value to the stack**, again. The model with all 30 features scores lower by itself than the one with only the expected values, and adds almost twice as much to the stack. In the final stack the two members with all 30 features get the largest weights of all 29 members (0.26 and 0.24), and most of our older XGBoost models go negative: the stack swaps them out.
- **Why would a function of the same columns help?** It adds no information. The expected values are computed from the same 20 columns the label model already sees, and with unlimited data a tree model could work them out. What changes is how hard that is: each one compresses 20 columns into a single number in the units of a rating. Our reading is that a rating in a synthetic row is one draw from a distribution that depends on the rest of the row, that the expected value is the centre of that distribution, and that the label follows the centre at least as much as the draw. That explanation is a guess. The ablation above is the evidence.

In [ ]:
X_aux_ev = pd.concat([X_og2, AUX[EV_COLS]], axis=1)          # the 13 expected values only
X_aux_all = pd.concat([X_og2, AUX], axis=1)                  # expected values, probabilities, row log-likelihood

for name, fit, X in [("xgb_auxev",  fit_xgb(),       X_aux_ev),
                     ("xgb_auxall", fit_xgb(seed=1), X_aux_all),
                     ("lgb_auxall", fit_lgb,         X_aux_all)]:
    run_cv(name, fit, X)

## 9. Stack

Logistic regression on the members' logits. AUC depends only on ranks, so the members are combined in logit space rather than as raw probabilities.
The stack's own score is cross-validated: each row is scored by weights fitted on the other rows.

The last lines show what each group of members is worth: the same stack refitted without that group.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

names = list(OOF)
Z = np.column_stack([logit(OOF[n]) for n in names])
ZT = np.column_stack([logit(TEST[n]) for n in names])

cv_score = np.zeros(NTR)
for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(Z, y):
    cv_score[b] = LogisticRegression(C=1.0, max_iter=1000).fit(Z[a], y[a]).decision_function(Z[b])
stack = LogisticRegression(C=1.0, max_iter=1000).fit(Z, y)

summary = pd.DataFrame({"OOF AUC": [roc_auc_score(y, OOF[n]) for n in names],
                        "stack weight": stack.coef_[0]}, index=names)
print(summary.round(6).to_string())
print(f"\nmean of logits       {roc_auc_score(y, Z.mean(1)):.6f}")
print(f"stack (cross-valid.) {roc_auc_score(y, cv_score):.6f}")

def stack_cv(keep):
    cols = [i for i, n in enumerate(names) if keep(n)]
    out = np.zeros(NTR)
    for a, b in StratifiedKFold(5, shuffle=True, random_state=7).split(Z, y):
        out[b] = LogisticRegression(C=1.0, max_iter=1000).fit(Z[a][:, cols], y[a]).decision_function(Z[b][:, cols])
    return roc_auc_score(y, out)
print(f"  without view B trees        {stack_cv(lambda n: not n.endswith('_vb')):.6f}")
print(f"  without flat_anneal RealMLP {stack_cv(lambda n: 'fa' not in n):.6f}")
NEW3 = ("cat_ctr", "cat_ctr3", "cat_ctr5", "glm_xgb", "glm6_xgb")
NEW4 = ("xgb_auxev", "xgb_auxall", "lgb_auxall")
print(f"  without auxiliary-task trees {stack_cv(lambda n: n not in NEW4):.6f}   <- the Part 3 stack (26)")
print(f"  without CatBoost CTR        {stack_cv(lambda n: not n.startswith('cat_ctr')):.6f}")
print(f"  without GLM-margin XGBoost  {stack_cv(lambda n: not n.startswith('glm')):.6f}")
print(f"  Part 2 members only (21)    {stack_cv(lambda n: n not in NEW3 + NEW4):.6f}")
print(f"  Part 1 members only (13)    {stack_cv(lambda n: n not in NEW3 + NEW4 and not n.endswith('_vb') and 'fa' not in n and n != 'deepfm_v1'):.6f}")

In [ ]:
pred = 1 / (1 + np.exp(-stack.decision_function(ZT)))
sub = pd.DataFrame({"id": te["id"].values, TARGET: pred})
sub.to_csv("submission.csv", index=False)

# member predictions, in case you want to blend them with your own models
pd.DataFrame({"id": tr["id"].values, **OOF}).to_csv("oof_members.csv", index=False)
pd.DataFrame({"id": te["id"].values, **TEST}).to_csv("test_members.csv", index=False)
sub.head()

## 10. A correction to Part 3: something was left to find

Part 3 (section 11) asked "is there anything left to find?" and answered no, on three checks. Each check was right about what it looked at, and none of them looked here:

- **The residual scan** grouped rows by single columns and by pairs of columns. An expected rating is a function of 20 columns at once. A scan over pairs cannot see it.
- **"Other people's models add nothing to our stack"** says that nobody had published this signal yet. It does not say the signal is absent.
- **"Five model families stop at the same place"**: all five were given the same two feature views.

The lesson we take: "we scanned and found nothing" has exactly the scope of the scan. It should have been written as "nothing left *in single columns and pairs*".

What still stands from Part 3: the public leaderboard cannot confirm a change of this size. This part moved it by +0.00002 (0.96127 → 0.96129). Part 3 moved it by the same +0.00002 with a smaller CV gain (+0.000035 against +0.000059). Everything above 0.9612 is one tie as far as the public leaderboard can tell; our decisions are made on paired 5-fold CV.

## 11. What did not work (new since Part 3)

| idea | result |
|---|---|
| The 16 "surprise" probabilities without the expected values | one XGBoost, fold 0: 0.961383 → 0.961284 |
| The row pseudo-log-likelihood alone | 0.961383 → 0.961404 (seed noise) |
| New public notebooks, checked for ideas to rebuild | blends of public submission files and baselines; nothing new to measure |

Still running as we publish, and reported in the next part whatever the outcome: CatBoost and RealMLP with the same features, view-B trees with them, the full predicted distribution of every column instead of its expected value, the residual (actual minus expected), and column predictors fitted on the original data only.

## 12. Where the earlier measurements are

- [Part 3](https://www.kaggle.com/code/goodpjw2008/s6e10-catboost-ctr-glm-margin-xgb-lb-0-96127): more folds raise CV but not the test score (held-out experiment), soft labels help one model but not a stack, the epoch-selection correction, and about thirty things that did not work.
- [Part 2](https://www.kaggle.com/code/goodpjw2008/s6e10-two-view-gbdt-realmlp-stack-lb-0-96125): the two feature views, why diversity is measured before it is built, noise-aware loss, zone models, pseudo-labels, the first generator replica.
- [Part 1](https://www.kaggle.com/code/goodpjw2008/s6e10-route-id-fe-og-model-stack-lb-0-96108): Flight Distance as a route ID, the original data as a model instead of as rows, rating aggregates, latent classes.

If this series helped you, please also upvote the notebooks in the credits sections of all four parts.